# ❤️ HEART DISEASE PREDICTOR
### ML Classification + Gemini AI + Streamlit

---

**What this project does:**
1. Takes the Cleveland Heart Disease dataset (just like adult.csv was income data)
2. Cleans it, does EDA, trains Logistic Regression + Random Forest
3. Launches a Streamlit app where:
   - User enters their health details
   - ML model predicts: **Heart Disease or No Heart Disease**
   - Gemini explains the result in simple words + gives diet/lifestyle tips

**Everything here is the same as your AICTE notebook — same libraries, same steps. Only the dataset and Streamlit UI are new.**

## STEP 1 — INSTALL LIBRARIES

In [ ]:
# Same libraries you already know
# Only new one: google-generativeai (you used this in AICTE_2 too!)
!pip install streamlit google-generativeai joblib -q

## STEP 2 — IMPORT LIBRARIES
Exact same imports as AICTE_1. Nothing new here.

In [ ]:
# ── SAME AS AICTE_1 ──────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

# ── SAME AS AICTE_1 ──────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ── NEW: to save and load the trained model ──────────────────
# joblib saves your trained model to a file so Streamlit can use it later
import joblib

print('✅ All libraries imported!')

## STEP 3 — LOAD DATASET

**About the dataset:**
- Called the **Cleveland Heart Disease dataset** (from Kaggle / UCI)
- 303 rows, 14 columns
- Target column: `target` → 1 = Heart Disease, 0 = No Heart Disease
- Just like adult.csv had an `income` column as target!

**Download from Kaggle:**
https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset

Upload the file to Colab, then run this cell.

In [ ]:
# ── SAME AS AICTE_1 ──────────────────────────────────────────
data = pd.read_csv('heart.csv')   # upload heart.csv to Colab first!

print('Shape:', data.shape)
print('\nFirst 3 rows:')
print(data.head(3))
print('\nLast 3 rows:')
print(data.tail(3))
print('\nColumn names:')
print(data.columns.tolist())

## STEP 4 — EDA (EXPLORATORY DATA ANALYSIS)

### What each column means (important for interview!)

| Column | Meaning |
|--------|----------|
| age | Age of patient |
| sex | 1 = Male, 0 = Female |
| cp | Chest pain type (0-3) |
| trestbps | Resting blood pressure |
| chol | Cholesterol level |
| fbs | Fasting blood sugar > 120 mg/dl (1=True, 0=False) |
| restecg | Resting ECG results (0-2) |
| thalach | Maximum heart rate achieved |
| exang | Exercise induced chest pain (1=Yes, 0=No) |
| oldpeak | ST depression |
| slope | Slope of peak exercise ST segment |
| ca | Number of major vessels (0-3) |
| thal | Thal type (0-3) |
| **target** | **1 = Heart Disease, 0 = No Heart Disease** |

In [ ]:
# ── NULL VALUE CHECK — SAME AS AICTE_1 ───────────────────────
print('── Null Values ──')
print(data.isna().sum())

# ── DUPLICATE CHECK — SAME AS AICTE_1 ────────────────────────
print('\n── Duplicates ──')
print(data.duplicated().sum())
data.drop_duplicates(inplace=True)
print('Duplicates removed ✅')

# ── BASIC STATS ───────────────────────────────────────────────
print('\n── Basic Statistics ──')
print(data.describe())

## STEP 5 — EDA VISUALIZATIONS
Same plots as AICTE_1 — distributions, correlation heatmap, target balance.

In [ ]:
# ── PLOT 1: TARGET BALANCE ────────────────────────────────────
# Shows how many people have heart disease vs don't
plt.figure(figsize=(5, 4))
sns.countplot(x='target', data=data, palette='Set2')
plt.title('Heart Disease Distribution')
plt.xticks([0, 1], ['No Disease', 'Heart Disease'])
plt.xlabel('Condition')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('target_distribution.png')
plt.show()
print('Saved target_distribution.png ✅')

In [ ]:
# ── PLOT 2: NUMERIC DISTRIBUTIONS — SAME AS AICTE_1 ──────────
numeric_cols = data.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'target']

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(data[col].dropna(), bins=20, color='#5B8DEF', edgecolor='white')
    axes[i].set_title(col)

# hide any extra blank plots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=14)
plt.tight_layout()
plt.savefig('distributions.png')
plt.show()
print('Saved distributions.png ✅')

In [ ]:
# ── PLOT 3: CORRELATION HEATMAP — SAME AS AICTE_1 ─────────────
plt.figure(figsize=(12, 8))
sns.heatmap(data.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation.png')
plt.show()
print('Saved correlation.png ✅')

In [ ]:
# ── PLOT 4: AGE vs HEART DISEASE ─────────────────────────────
# Extra insight: do older people have more heart disease?
plt.figure(figsize=(8, 4))
sns.boxplot(x='target', y='age', data=data, palette='Set2')
plt.title('Age vs Heart Disease')
plt.xticks([0, 1], ['No Disease', 'Heart Disease'])
plt.tight_layout()
plt.savefig('age_vs_target.png')
plt.show()
print('Saved age_vs_target.png ✅')

## STEP 6 — DATA CLEANING
Same 3 steps as AICTE_1: fill nulls → fix outliers → encode categories.

In [ ]:
# ── STEP 6a: FILL NULL VALUES — SAME AS AICTE_1 ──────────────
for col in data.select_dtypes(include=np.number).columns:
    if data[col].isnull().sum() > 0:
        data[col].fillna(data[col].median(), inplace=True)
        print(f"Filled null in '{col}' with median ✅")

print('Null check after fill:', data.isnull().sum().sum(), 'nulls remaining')

In [ ]:
# ── STEP 6b: OUTLIER CLIPPING — SAME AS AICTE_1 ──────────────
# Using the same IQR method you learnt
for col in data.select_dtypes(include=np.number).columns:
    if col == 'target':
        continue   # never clip the target column!
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((data[col] < lower) | (data[col] > upper)).sum()
    if outliers > 0:
        data[col] = data[col].clip(lower=lower, upper=upper)
        print(f"Fixed outliers in '{col}' ✅")

print('\nData shape after cleaning:', data.shape)

In [ ]:
# ── STEP 6c: LABEL ENCODE CATEGORICAL COLUMNS — SAME AS AICTE_1
# This dataset is mostly numeric already, but we handle any object columns
le = LabelEncoder()
for col in data.select_dtypes(include='object').columns:
    data[col] = le.fit_transform(data[col])
    print(f"Encoded '{col}' ✅")

print('\n✅ Cleaning complete!')
print(data.head(3))

## STEP 7 — TRAIN/TEST SPLIT + MODEL TRAINING
100% same as AICTE_1. Only difference: target column is `target` instead of `income`.

In [ ]:
# ── SAME AS AICTE_1 — just target column name changed ─────────
X = data.drop(columns=['target'])
y = data['target']

# Train-test split 80/20 — same as AICTE_1
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing samples  : {X_test.shape[0]}')

# ── LOGISTIC REGRESSION — SAME AS AICTE_1 ────────────────────
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

# ── RANDOM FOREST — SAME AS AICTE_1 ──────────────────────────
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print('\n✅ Both models trained!')

## STEP 8 — MODEL EVALUATION
Accuracy, classification report, confusion matrix — exact same as AICTE_1.

In [ ]:
# ── SAME AS AICTE_1 ──────────────────────────────────────────
print('=' * 50)
print('  LOGISTIC REGRESSION RESULTS')
print('=' * 50)
print(f'Accuracy : {accuracy_score(y_test, lr_pred)*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, lr_pred,
      target_names=['No Disease', 'Heart Disease']))

print('=' * 50)
print('  RANDOM FOREST RESULTS')
print('=' * 50)
print(f'Accuracy : {accuracy_score(y_test, rf_pred)*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, rf_pred,
      target_names=['No Disease', 'Heart Disease']))

In [ ]:
# ── CONFUSION MATRIX — SAME AS AICTE_1 ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(confusion_matrix(y_test, lr_pred),
            annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Disease', 'Heart Disease'],
            yticklabels=['No Disease', 'Heart Disease'])
axes[0].set_title('Logistic Regression — Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

sns.heatmap(confusion_matrix(y_test, rf_pred),
            annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['No Disease', 'Heart Disease'],
            yticklabels=['No Disease', 'Heart Disease'])
axes[1].set_title('Random Forest — Confusion Matrix')
axes[1].set_ylabel('Actual')
axes[1].set_xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_matrices.png')
plt.show()
print('Saved confusion_matrices.png ✅')

In [ ]:
# ── FEATURE IMPORTANCE — SAME AS AICTE_1 ─────────────────────
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns)
feat_imp = feat_imp.sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
plt.title('Feature Importance — Random Forest')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.show()

print('\n── Top 5 Most Important Features ──')
print(feat_imp.head(5))

## STEP 9 — SAVE THE MODEL

**Why save the model?**

Your Streamlit app runs separately from this notebook.
So you train the model here → save it as a file → Streamlit loads that file.

Think of it like baking a cake (training) and storing it in a box (joblib.dump).
The app just opens the box (joblib.load) and serves it!

We save **Random Forest** because it had better accuracy.

In [ ]:
# ── SAVE MODEL — NEW but simple ───────────────────────────────
# joblib.dump(what_to_save, 'filename')
joblib.dump(rf_model, 'heart_model.pkl')

# Also save the feature column names
# So the app knows which columns to expect in the right order
joblib.dump(X.columns.tolist(), 'feature_names.pkl')

print('Model saved as heart_model.pkl ✅')
print('Feature names saved as feature_names.pkl ✅')
print('\nRandom Forest Accuracy:', f"{accuracy_score(y_test, rf_pred)*100:.2f}%")

## STEP 10 — STREAMLIT APP

**How this works (same as AICTE_2):**
- `%%writefile app.py` saves everything below into a file called `app.py`
- Then we run that file with streamlit

**The app has 2 tabs (same as AICTE_2 had tabs):**
- Tab 1: Enter health details → ML prediction + Gemini explanation
- Tab 2: Gemini gives diet/lifestyle tips based on what you entered

**Replace `YOUR_GEMINI_API_KEY` with your actual key before running!**

In [ ]:
%%writefile app.py
# ============================================================
# HEART DISEASE PREDICTOR — STREAMLIT APP
# Uses: ML model (Random Forest) + Gemini AI
# ============================================================

import streamlit as st
import numpy as np
import joblib
import google.generativeai as genai

# ── SETUP ────────────────────────────────────────────────────
# Same as AICTE_2 — configure Gemini with your API key
genai.configure(api_key="YOUR_GEMINI_API_KEY")
model_gemini = genai.GenerativeModel("gemini-1.5-flash")

# Load the saved ML model and feature names
# joblib.load is the opposite of joblib.dump — opens the saved box!
rf_model = joblib.load('heart_model.pkl')
feature_names = joblib.load('feature_names.pkl')

# ── PAGE SETUP ───────────────────────────────────────────────
# Same as AICTE_2
st.set_page_config(
    page_title="Heart Disease Predictor",
    page_icon="❤️",
    layout="wide"
)

st.title("❤️ Heart Disease Predictor")
st.write("Enter your health details below. The ML model will predict your risk, and Gemini AI will explain the result.")
st.markdown("---")

# ── 2 TABS — same concept as AICTE_2 ────────────────────────
tab1, tab2 = st.tabs(["🔬 Prediction", "🥗 Diet & Lifestyle Tips"])


# ════════════════════════════════════════════════════════════
# TAB 1 — PREDICTION
# User fills in their health values → model predicts → Gemini explains
# ════════════════════════════════════════════════════════════
with tab1:
    st.subheader("Enter Your Health Details")

    # ── INPUT FORM ───────────────────────────────────────────
    # Two columns so it doesn't look too long
    col1, col2 = st.columns(2)

    with col1:
        age      = st.number_input("Age", min_value=20, max_value=100, value=45)
        sex      = st.selectbox("Sex", options=[1, 0],
                                format_func=lambda x: "Male" if x == 1 else "Female")
        cp       = st.selectbox("Chest Pain Type",
                                options=[0, 1, 2, 3],
                                format_func=lambda x: [
                                    "0 — Typical angina",
                                    "1 — Atypical angina",
                                    "2 — Non-anginal pain",
                                    "3 — Asymptomatic"
                                ][x])
        trestbps = st.number_input("Resting Blood Pressure (mm Hg)",
                                   min_value=80, max_value=200, value=120)
        chol     = st.number_input("Cholesterol (mg/dl)",
                                   min_value=100, max_value=600, value=200)
        fbs      = st.selectbox("Fasting Blood Sugar > 120 mg/dl",
                                options=[0, 1],
                                format_func=lambda x: "Yes" if x == 1 else "No")
        restecg  = st.selectbox("Resting ECG",
                                options=[0, 1, 2],
                                format_func=lambda x: [
                                    "0 — Normal",
                                    "1 — ST-T wave abnormality",
                                    "2 — Left ventricular hypertrophy"
                                ][x])

    with col2:
        thalach  = st.number_input("Max Heart Rate Achieved",
                                   min_value=60, max_value=220, value=150)
        exang    = st.selectbox("Exercise Induced Chest Pain",
                                options=[0, 1],
                                format_func=lambda x: "Yes" if x == 1 else "No")
        oldpeak  = st.number_input("ST Depression (oldpeak)",
                                   min_value=0.0, max_value=7.0,
                                   value=1.0, step=0.1)
        slope    = st.selectbox("Slope of Peak Exercise ST Segment",
                                options=[0, 1, 2],
                                format_func=lambda x: [
                                    "0 — Upsloping",
                                    "1 — Flat",
                                    "2 — Downsloping"
                                ][x])
        ca       = st.selectbox("Number of Major Vessels (0-3)",
                                options=[0, 1, 2, 3])
        thal     = st.selectbox("Thal",
                                options=[0, 1, 2, 3],
                                format_func=lambda x: [
                                    "0 — Normal",
                                    "1 — Fixed defect",
                                    "2 — Reversible defect",
                                    "3 — Other"
                                ][x])

    # ── PREDICT BUTTON ───────────────────────────────────────
    if st.button("🔍 Predict Now", use_container_width=True):

        # Build input array in the same column order as training
        # This is why we saved feature_names.pkl !
        input_data = np.array([[age, sex, cp, trestbps, chol, fbs,
                                 restecg, thalach, exang, oldpeak,
                                 slope, ca, thal]])

        # Get prediction (0 or 1) and probability
        prediction   = rf_model.predict(input_data)[0]
        probability  = rf_model.predict_proba(input_data)[0]
        confidence   = probability[prediction] * 100

        # ── SHOW RESULT ──────────────────────────────────────
        st.markdown("---")
        if prediction == 1:
            st.error(f"⚠️ **Heart Disease Detected** — Confidence: {confidence:.1f}%")
        else:
            st.success(f"✅ **No Heart Disease Detected** — Confidence: {confidence:.1f}%")

        # ── GEMINI EXPLANATION ───────────────────────────────
        # Same pattern as AICTE_2 — build a prompt, send to Gemini
        result_text = "Heart Disease" if prediction == 1 else "No Heart Disease"

        prompt = f"""
        A patient has been assessed with these health details:
        - Age: {age}, Sex: {'Male' if sex==1 else 'Female'}
        - Chest Pain Type: {cp}, Resting BP: {trestbps} mm Hg
        - Cholesterol: {chol} mg/dl, Fasting Blood Sugar >120: {'Yes' if fbs==1 else 'No'}
        - Max Heart Rate: {thalach}, Exercise Chest Pain: {'Yes' if exang==1 else 'No'}
        - ST Depression: {oldpeak}, Major Vessels: {ca}

        The ML model predicted: **{result_text}** with {confidence:.1f}% confidence.

        Please:
        1. Explain this result in very simple words (2-3 sentences, no medical jargon)
        2. Point out which 2-3 values from their inputs seem most concerning or reassuring
        3. End with: "Please consult a doctor for proper medical advice."

        Keep the tone calm and supportive. Use bullet points.
        """

        with st.spinner("Gemini is analyzing your results..."):
            response = model_gemini.generate_content(prompt)

        st.markdown("### 🤖 Gemini's Explanation")
        st.markdown(response.text)

        # Store inputs in session_state so Tab 2 can use them
        # session_state = Streamlit's way of passing data between tabs
        st.session_state['inputs'] = {
            'age': age, 'sex': sex, 'chol': chol,
            'trestbps': trestbps, 'fbs': fbs,
            'thalach': thalach, 'prediction': result_text
        }

        st.info("💡 Go to the **Diet & Lifestyle Tips** tab for personalised advice!")


# ════════════════════════════════════════════════════════════
# TAB 2 — DIET & LIFESTYLE TIPS
# Gemini gives personalised tips based on the user's inputs
# ════════════════════════════════════════════════════════════
with tab2:
    st.subheader("🥗 Personalised Diet & Lifestyle Tips")

    # Check if user has done a prediction yet
    if 'inputs' not in st.session_state:
        st.info("👈 First go to the Prediction tab and click Predict Now. Then come back here!")
    else:
        inp = st.session_state['inputs']

        st.markdown(f"**Based on your results: {inp['prediction']}**")
        st.markdown("Gemini will now give you personalised tips based on your specific values.")

        if st.button("🥗 Get My Tips", use_container_width=True):

            tips_prompt = f"""
            A patient with these details:
            - Age: {inp['age']}, Sex: {'Male' if inp['sex']==1 else 'Female'}
            - Cholesterol: {inp['chol']} mg/dl
            - Resting Blood Pressure: {inp['trestbps']} mm Hg
            - Fasting Blood Sugar >120: {'Yes' if inp['fbs']==1 else 'No'}
            - Max Heart Rate: {inp['thalach']}
            - ML Prediction: {inp['prediction']}

            Give them a friendly, personalised health plan with:
            1. 🥗 3 specific diet tips (mention actual foods based on their cholesterol/BP values)
            2. 🏃 3 specific exercise tips suited to their age and heart rate
            3. 😴 2 lifestyle habits to adopt
            4. ⚠️ 2 things they should strictly avoid

            Use simple language, be encouraging, and use emojis.
            End with a reminder to consult a doctor.
            """

            with st.spinner("Gemini is preparing your personalised plan..."):
                tips_response = model_gemini.generate_content(tips_prompt)

            st.markdown(tips_response.text)

## STEP 11 — RUN THE APP

Same exact method as AICTE_2 — kill old streamlit, start new one, open with localtunnel.

**You will get a URL like:** `https://xxxx.loca.lt`
Open that URL → your app is live!

In [ ]:
# ── SAME AS AICTE_2 ──────────────────────────────────────────
!pkill -f streamlit

import subprocess, time
time.sleep(2)

subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port=8501'])
time.sleep(3)

print('✅ Streamlit is running!')
print('Opening tunnel...')

!npx --yes localtunnel --port 8501

---
## ✅ PROJECT COMPLETE!

### What you built:
- ✅ Loaded and explored real heart disease data
- ✅ Cleaned it (nulls, duplicates, outliers) — same as AICTE_1
- ✅ Trained Logistic Regression + Random Forest — same as AICTE_1
- ✅ Evaluated with accuracy, F1, confusion matrix — same as AICTE_1
- ✅ Saved the model with joblib
- ✅ Built a Streamlit app with 2 tabs — same as AICTE_2
- ✅ Integrated Gemini AI to explain predictions + give health tips

### What to say in interviews:
> *"I built a Heart Disease Predictor that combines ML classification using Random Forest with Google Gemini AI. The ML model predicts risk from clinical data, and Gemini explains the result in simple terms and gives personalised diet and lifestyle tips. I deployed it as a web app using Streamlit."*

### Interview questions you can now answer:
- Why Random Forest over Logistic Regression? → *"RF gave better accuracy on this dataset"*
- What does your confusion matrix show? → *"True positives, false positives — how many disease cases the model correctly caught"*
- What were the most important features? → *"cp (chest pain type), thalach (max heart rate), ca (major vessels) — shown in feature importance plot"*
- How did you handle missing values? → *"Filled numeric columns with median, same IQR method for outliers"*